In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
import secrets
import string


DATE_FORMAT = "%d/%m/%Y"
ADMIN_PASSWORD = "admin1989"  # Change this before using the program in real life.


def valid_date(value):
    """Return True when value is a valid date in dd/mm/yyyy format."""
    try:
        datetime.strptime(value, DATE_FORMAT)
        return True
    except ValueError:
        return False


def ask_date(prompt):
    while True:
        value = input(prompt).strip()
        if valid_date(value):
            return value
        print("Invalid date. Please use dd/mm/yyyy (for example, 05/10/2002).")


def ask_number(prompt, number_type=float, minimum=0):
    while True:
        try:
            value = number_type(input(prompt).strip())
            if value < minimum:
                print(f"Please enter a value greater than or equal to {minimum}.")
                continue
            return value
        except ValueError:
            print("Invalid number. Please try again.")


@dataclass
class Subscription:
    package_name: str
    price: float
    num_classes_per_week: int
    num_private_sessions: int


@dataclass
class Attendance:
    member: "Member"
    date: str
    status: str = "Present"


@dataclass
class Trainer:
    name: str
    specialization: str
    working_time: str


@dataclass
class Staff:
    name: str
    designation: str
    working_time: str


@dataclass
class Schedule:
    fitness_program: str
    trainers: list = field(default_factory=list)
    # Each private session is a dictionary containing a date and trainer.
    private_sessions: list = field(default_factory=list)

    def add_private_session(self, date, trainer):
        self.private_sessions.append({"date": date, "trainer": trainer})


@dataclass
class Member:
    full_name: str
    mobile_number: str
    dob: str
    date_of_joining: str
    subscription: Subscription = None
    username: str = ""
    password: str = ""
    attendance: list = field(default_factory=list)

    def view_info(self):
        print("\nMember Information")
        print(f"Full Name: {self.full_name}")
        print(f"Mobile Number: {self.mobile_number}")
        print(f"Date of Birth: {self.dob}")
        print(f"Date of Joining Gym: {self.date_of_joining}")
        print(f"Username: {self.username or 'Not set'}")

    def view_attendance(self):
        print("\nAttendance")
        if not self.attendance:
            print("No attendance records found.")
            return
        for record in self.attendance:
            print(f"Date: {record.date}, Status: {record.status}")

    def change_password(self):
        new_password = input("Enter your new password (at least 6 characters): ").strip()
        while len(new_password) < 6:
            print("Password must contain at least 6 characters.")
            new_password = input("Enter your new password: ").strip()
        self.password = new_password
        print("Password changed successfully.")


class Gym:
    def __init__(self):
        self.members = []
        self.trainers = []
        self.staff = []
        self.schedules = []
        self.attendance = []
        self.subscription_packages = [
            Subscription("Lite (Weekly)", 10, 0, 0),
            Subscription("Basic (Weekly)", 25, 1, 0),
            Subscription("Premium (Monthly)", 100, 2, 10),
            Subscription("PremiumPlus (Yearly)", 1150, 5, 50),
        ]


# ---------- Shared lookup helpers ----------
def find_member_by_mobile_number(gym, mobile_number):
    for member in gym.members:
        if member.mobile_number == mobile_number:
            return member
    return None


def find_trainer_by_name(gym, name):
    for trainer in gym.trainers:
        if trainer.name.casefold() == name.casefold():
            return trainer
    return None


def find_staff_by_name(gym, name):
    for staff in gym.staff:
        if staff.name.casefold() == name.casefold():
            return staff
    return None


def find_subscription_package_by_name(gym, package_name):
    for package in gym.subscription_packages:
        if package.package_name.casefold() == package_name.casefold():
            return package
    return None


def find_schedule_by_fitness_program(gym, fitness_program):
    for schedule in gym.schedules:
        if schedule.fitness_program.casefold() == fitness_program.casefold():
            return schedule
    return None


def generate_random_password(length=8):
    characters = string.ascii_letters + string.digits
    return "".join(secrets.choice(characters) for _ in range(length))


def send_sms(mobile_number, message):
    # Demo only: connect an SMS provider here to send real messages.
    print(f"\n[SMS DEMO to {mobile_number}] {message}")


# ---------- Members ----------
def add_member(gym):
    print("\nAdd Member")
    full_name = input("Full Name: ").strip()
    mobile_number = input("Mobile Number: ").strip()
    if not full_name or not mobile_number:
        print("Name and mobile number cannot be empty.")
        return
    if find_member_by_mobile_number(gym, mobile_number):
        print("A member with that mobile number already exists.")
        return

    dob = ask_date("Date of Birth (dd/mm/yyyy): ")
    date_of_joining = ask_date("Date of Joining Gym (dd/mm/yyyy): ")
    show_subscription_packages(gym)
    package_choice = input("Choose a package number, or press Enter for no package: ").strip()

    subscription = None
    if package_choice:
        if not package_choice.isdigit() or not (1 <= int(package_choice) <= len(gym.subscription_packages)):
            print("Invalid package choice. Member will be added without a subscription.")
        else:
            subscription = gym.subscription_packages[int(package_choice) - 1]

    username = full_name.split()[0].lower()
    password = generate_random_password()
    member = Member(full_name, mobile_number, dob, date_of_joining,
                    subscription=subscription, username=username, password=password)
    gym.members.append(member)

    if subscription:
        print(f"Member added successfully with package: {subscription.package_name}.")
        send_sms(mobile_number,
                 f"Dear {full_name}, your username is {username} and password is {password}. "
                 "Please make a payment within 3 days to complete your membership.")
    else:
        print("Member added successfully without a subscription package.")
        print(f"Temporary login username: {username}; password: {password}")


def remove_member(gym):
    mobile_number = input("Enter the mobile number of the member to remove: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if not member:
        print("Member not found.")
        return
    gym.members.remove(member)
    # Keep the central attendance list consistent when a member is removed.
    gym.attendance = [record for record in gym.attendance if record.member is not member]
    print("Member removed successfully.")


def update_member_information(gym):
    mobile_number = input("Enter the mobile number of the member to update: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if not member:
        print("Member not found.")
        return
    full_name = input(f"Full Name [{member.full_name}]: ").strip()
    dob = input(f"Date of Birth [{member.dob}] (dd/mm/yyyy): ").strip()
    if full_name:
        member.full_name = full_name
    if dob:
        if not valid_date(dob):
            print("Invalid date. Date of birth was not changed.")
        else:
            member.dob = dob
    print("Member information updated.")


def view_member_information(gym):
    if not gym.members:
        print("No members found.")
        return
    print("\nAll Members")
    for member in gym.members:
        print(f"Name: {member.full_name}, Mobile: {member.mobile_number}")


def view_member_information_by_mobile(gym, mobile_number=None):
    if mobile_number is None:
        mobile_number = input("Enter the mobile number of the member: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if member:
        member.view_info()
    else:
        print("Member not found.")


def view_member_attendance(gym, mobile_number=None):
    if mobile_number is None:
        mobile_number = input("Enter the mobile number of the member: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if member:
        member.view_attendance()
    else:
        print("Member not found.")


def view_subscription_details(gym, mobile_number=None):
    if mobile_number is None:
        mobile_number = input("Enter the mobile number of the member: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if not member:
        print("Member not found.")
    elif not member.subscription:
        print("Member does not have an active subscription.")
    else:
        package = member.subscription
        print("\nSubscription Details")
        print(f"Package: {package.package_name}")
        print(f"Price: ${package.price:.2f}")
        print(f"Classes per Week: {package.num_classes_per_week}")
        print(f"Private Sessions: {package.num_private_sessions}")


# ---------- Trainers ----------
def add_trainer(gym):
    name = input("Trainer Name: ").strip()
    specialization = input("Trainer Specialization: ").strip()
    working_time = input("Working Time: ").strip()
    if not name:
        print("Trainer name cannot be empty.")
        return
    if find_trainer_by_name(gym, name):
        print("A trainer with that name already exists.")
        return
    gym.trainers.append(Trainer(name, specialization, working_time))
    print("Trainer added successfully.")


def remove_trainer(gym):
    name = input("Enter the name of the trainer to remove: ").strip()
    trainer = find_trainer_by_name(gym, name)
    if trainer:
        gym.trainers.remove(trainer)
        print("Trainer removed successfully.")
    else:
        print("Trainer not found.")


def update_trainer_information(gym):
    name = input("Enter the name of the trainer to update: ").strip()
    trainer = find_trainer_by_name(gym, name)
    if not trainer:
        print("Trainer not found.")
        return
    specialization = input(f"Specialization [{trainer.specialization}]: ").strip()
    working_time = input(f"Working Time [{trainer.working_time}]: ").strip()
    if specialization:
        trainer.specialization = specialization
    if working_time:
        trainer.working_time = working_time
    print("Trainer information updated.")


def view_trainer_information(gym):
    print("\nAll Trainers")
    if not gym.trainers:
        print("No trainers found.")
        return
    for trainer in gym.trainers:
        print(f"Name: {trainer.name}, Specialization: {trainer.specialization}, Working Time: {trainer.working_time}")


# ---------- Staff ----------
def add_staff(gym):
    name = input("Staff Name: ").strip()
    designation = input("Designation: ").strip()
    working_time = input("Working Time: ").strip()
    if not name:
        print("Staff name cannot be empty.")
        return
    if find_staff_by_name(gym, name):
        print("A staff member with that name already exists.")
        return
    gym.staff.append(Staff(name, designation, working_time))
    print("Staff added successfully.")


def remove_staff(gym):
    name = input("Enter the name of the staff member to remove: ").strip()
    staff = find_staff_by_name(gym, name)
    if staff:
        gym.staff.remove(staff)
        print("Staff removed successfully.")
    else:
        print("Staff member not found.")


def update_staff_information(gym):
    name = input("Enter the name of the staff member to update: ").strip()
    staff = find_staff_by_name(gym, name)
    if not staff:
        print("Staff member not found.")
        return
    designation = input(f"Designation [{staff.designation}]: ").strip()
    working_time = input(f"Working Time [{staff.working_time}]: ").strip()
    if designation:
        staff.designation = designation
    if working_time:
        staff.working_time = working_time
    print("Staff information updated.")


def view_staff_information(gym):
    print("\nAll Staff")
    if not gym.staff:
        print("No staff found.")
        return
    for staff in gym.staff:
        print(f"Name: {staff.name}, Designation: {staff.designation}, Working Time: {staff.working_time}")


# ---------- Subscription packages ----------
def show_subscription_packages(gym):
    print("\nSubscription Packages")
    if not gym.subscription_packages:
        print("No subscription packages available.")
        return
    for index, package in enumerate(gym.subscription_packages, start=1):
        print(f"{index}. {package.package_name} - ${package.price:.2f}; "
              f"classes/week: {package.num_classes_per_week}; "
              f"private sessions: {package.num_private_sessions}")


def add_subscription_package(gym):
    package_name = input("Enter the name of the subscription package: ").strip()
    if not package_name:
        print("Package name cannot be empty.")
        return
    if find_subscription_package_by_name(gym, package_name):
        print("A package with that name already exists.")
        return
    price = ask_number("Enter the price: $", float, 0)
    classes = ask_number("Enter the number of classes per week: ", int, 0)
    sessions = ask_number("Enter the number of private sessions: ", int, 0)
    gym.subscription_packages.append(Subscription(package_name, price, classes, sessions))
    print("Subscription package added successfully.")


def remove_subscription_package(gym):
    package_name = input("Enter the name of the package to remove: ").strip()
    package = find_subscription_package_by_name(gym, package_name)
    if not package:
        print("Subscription package not found.")
        return
    gym.subscription_packages.remove(package)
    print("Subscription package removed. Existing members keep their assigned package details.")


def update_subscription_package(gym):
    package_name = input("Enter the name of the package to update: ").strip()
    package = find_subscription_package_by_name(gym, package_name)
    if not package:
        print("Subscription package not found.")
        return
    price = ask_number(f"New price [${package.price:.2f}]: $", float, 0)
    classes = ask_number(f"New classes per week [{package.num_classes_per_week}]: ", int, 0)
    sessions = ask_number(f"New private sessions [{package.num_private_sessions}]: ", int, 0)
    package.price = price
    package.num_classes_per_week = classes
    package.num_private_sessions = sessions
    print("Subscription package updated successfully.")


def view_subscription_packages(gym):
    show_subscription_packages(gym)


# ---------- Schedules and private sessions ----------
def add_schedule(gym):
    fitness_program = input("Enter the name of the fitness program: ").strip()
    if not fitness_program:
        print("Fitness program name cannot be empty.")
        return
    if find_schedule_by_fitness_program(gym, fitness_program):
        print("A schedule for that program already exists.")
        return
    if len(gym.trainers) < 2:
        print("Add at least two trainers before creating a schedule.")
        return
    view_trainer_information(gym)
    trainer_names = []
    trainers = []
    for i in range(2):
        name = input(f"Enter the name of trainer {i + 1}: ").strip()
        trainer = find_trainer_by_name(gym, name)
        if not trainer:
            print("Trainer not found. Schedule was not created.")
            return
        if trainer in trainers:
            print("Please choose two different trainers.")
            return
        trainer_names.append(name)
        trainers.append(trainer)
    gym.schedules.append(Schedule(fitness_program, trainers))
    print("Schedule added successfully.")


def remove_schedule(gym):
    name = input("Enter the fitness program to remove: ").strip()
    schedule = find_schedule_by_fitness_program(gym, name)
    if schedule:
        gym.schedules.remove(schedule)
        print("Schedule removed successfully.")
    else:
        print("Schedule not found.")


def update_schedule(gym):
    name = input("Enter the fitness program to update: ").strip()
    schedule = find_schedule_by_fitness_program(gym, name)
    if not schedule:
        print("Schedule not found.")
        return
    new_name = input(f"New fitness program name [{schedule.fitness_program}]: ").strip()
    if new_name:
        existing = find_schedule_by_fitness_program(gym, new_name)
        if existing and existing is not schedule:
            print("A schedule with that name already exists.")
            return
        schedule.fitness_program = new_name
    print("Schedule updated successfully.")


def view_schedule(gym):
    print("\nAll Schedules")
    if not gym.schedules:
        print("No schedules found.")
        return
    for schedule in gym.schedules:
        print(f"Fitness Program: {schedule.fitness_program}")
        print("Trainers: " + ", ".join(trainer.name for trainer in schedule.trainers))


def add_private_session(gym):
    name = input("Enter the fitness program: ").strip()
    schedule = find_schedule_by_fitness_program(gym, name)
    if not schedule:
        print("Schedule not found.")
        return
    date = ask_date("Enter the date of the private session (dd/mm/yyyy): ")
    trainer_name = input("Enter the trainer's name: ").strip()
    trainer = find_trainer_by_name(gym, trainer_name)
    if not trainer:
        print("Trainer not found.")
        return
    if trainer not in schedule.trainers:
        print("That trainer is not assigned to this fitness program.")
        return
    schedule.add_private_session(date, trainer)
    print("Private session added successfully.")


def remove_private_session(gym):
    name = input("Enter the fitness program: ").strip()
    schedule = find_schedule_by_fitness_program(gym, name)
    if not schedule:
        print("Schedule not found.")
        return
    if not schedule.private_sessions:
        print("This schedule has no private sessions.")
        return
    view_private_session_schedule(gym, only_schedule=schedule)
    date = ask_date("Enter the date of the private session to remove (dd/mm/yyyy): ")
    matches = [session for session in schedule.private_sessions if session["date"] == date]
    if not matches:
        print("Private session not found on that date.")
        return
    if len(matches) > 1:
        print("More than one session exists on that date. Enter the trainer's name too.")
        trainer_name = input("Trainer name: ").strip()
        matches = [session for session in matches if session["trainer"].name.casefold() == trainer_name.casefold()]
    if matches:
        schedule.private_sessions.remove(matches[0])
        print("Private session removed successfully.")
    else:
        print("Matching private session not found.")


def view_private_session_schedule(gym, only_schedule=None):
    print("\nPrivate Session Schedule")
    schedules = [only_schedule] if only_schedule else gym.schedules
    if not schedules:
        print("No schedules found.")
        return
    for schedule in schedules:
        print(f"Fitness Program: {schedule.fitness_program}")
        if schedule.private_sessions:
            for session in schedule.private_sessions:
                print(f"- Date: {session['date']}, Trainer: {session['trainer'].name}")
        else:
            print("No private sessions for this schedule.")


# ---------- Attendance ----------
def mark_attendance(gym):
    mobile_number = input("Enter the mobile number of the member: ").strip()
    member = find_member_by_mobile_number(gym, mobile_number)
    if not member:
        print("Member not found.")
        return
    date = ask_date("Enter the date of attendance (dd/mm/yyyy): ")
    status = input("Enter attendance status (Present/Absent): ").strip().capitalize()
    if status not in ("Present", "Absent"):
        print("Invalid attendance status. Use Present or Absent.")
        return
    # Update an existing record for this member/date instead of duplicating it.
    for record in member.attendance:
        if record.date == date:
            record.status = status
            print("Existing attendance record updated.")
            return
    record = Attendance(member, date, status)
    member.attendance.append(record)
    gym.attendance.append(record)
    print("Attendance marked successfully.")


# ---------- Menus ----------
def display_menu(user_type):
    if user_type == "admin":
        print("\nAdmin Menu")
        options = [
            "0. Exit", "1. Add Member", "2. Remove Member", "3. Update Member Information",
            "4. View All Members", "5. Add Trainer", "6. Remove Trainer", "7. Update Trainer Information",
            "8. View Trainers", "9. Add Staff", "10. Remove Staff", "11. Update Staff Information",
            "12. View Staff", "13. Add Subscription Package", "14. Remove Subscription Package",
            "15. Update Subscription Package", "16. View Subscription Packages", "17. Add Schedule",
            "18. Remove Schedule", "19. Update Schedule", "20. View Schedules",
            "21. Add Private Session", "22. Remove Private Session", "23. View Private Sessions",
            "24. Mark Attendance",
        ]
    else:
        print("\nMember Menu")
        options = [
            "0. Exit", "1. View My Information", "2. View My Attendance",
            "3. View Subscription Details", "4. Change Password",
        ]
    for option in options:
        print(option)


def run_admin_choice(gym, choice):
    actions = {
        "1": lambda: add_member(gym),
        "2": lambda: remove_member(gym),
        "3": lambda: update_member_information(gym),
        "4": lambda: view_member_information(gym),
        "5": lambda: add_trainer(gym),
        "6": lambda: remove_trainer(gym),
        "7": lambda: update_trainer_information(gym),
        "8": lambda: view_trainer_information(gym),
        "9": lambda: add_staff(gym),
        "10": lambda: remove_staff(gym),
        "11": lambda: update_staff_information(gym),
        "12": lambda: view_staff_information(gym),
        "13": lambda: add_subscription_package(gym),
        "14": lambda: remove_subscription_package(gym),
        "15": lambda: update_subscription_package(gym),
        "16": lambda: view_subscription_packages(gym),
        "17": lambda: add_schedule(gym),
        "18": lambda: remove_schedule(gym),
        "19": lambda: update_schedule(gym),
        "20": lambda: view_schedule(gym),
        "21": lambda: add_private_session(gym),
        "22": lambda: remove_private_session(gym),
        "23": lambda: view_private_session_schedule(gym),
        "24": lambda: mark_attendance(gym),
    }
    action = actions.get(choice)
    if action:
        action()
    elif choice != "0":
        print("Invalid choice. Try again.")


def main():
    gym = Gym()

    # Optional sample data for testing. Remove these examples if you want an empty gym.
    sample_member = Member("Nafid Bin Anis", "01303362078", "05/10/2002", "31/08/2023",
                           subscription=gym.subscription_packages[1], username="nafid", password="nafid123")
    gym.members.append(sample_member)
    gym.trainers.append(Trainer("Farhanul Islam", "Yoga", "10:00 AM - 2:00 PM"))
    gym.trainers.append(Trainer("Sample Trainer", "Strength Training", "2:00 PM - 6:00 PM"))
    gym.staff.append(Staff("Julkar Junain", "Receptionist", "9:00 AM - 5:00 PM"))

    while True:
        user_type = input("\nEnter 'admin' for Admin Menu, 'member' for Member Menu, or 'exit' to quit: ").strip().lower()
        if user_type == "exit":
            print("Goodbye!")
            break

        if user_type == "admin":
            admin_password = input("Enter the admin password: ")
            if admin_password != ADMIN_PASSWORD:
                print("Incorrect admin password.")
                continue
            while True:
                display_menu("admin")
                choice = input("Enter your choice: ").strip()
                if choice == "0":
                    break
                run_admin_choice(gym, choice)

        elif user_type == "member":
            mobile_number = input("Enter your mobile number: ").strip()
            member = find_member_by_mobile_number(gym, mobile_number)
            if not member:
                print("Member not found.")
                continue
            password = input("Enter your password: ")
            if password != member.password:
                print("Incorrect password.")
                continue
            while True:
                display_menu("member")
                choice = input("Enter your choice: ").strip()
                if choice == "0":
                    break
                elif choice == "1":
                    member.view_info()
                elif choice == "2":
                    member.view_attendance()
                elif choice == "3":
                    if member.subscription:
                        package = member.subscription
                        print(f"Package: {package.package_name}\nPrice: ${package.price:.2f}\n"
                              f"Classes per Week: {package.num_classes_per_week}\n"
                              f"Private Sessions: {package.num_private_sessions}")
                    else:
                        print("You do not have an active subscription.")
                elif choice == "4":
                    member.change_password()
                else:
                    print("Invalid choice. Try again.")
        else:
            print("Invalid user type. Try again.")


if __name__ == "__main__":
    main()



Enter 'admin' for Admin Menu, 'member' for Member Menu, or 'exit' to quit: admin
Enter the admin password: admin1989

Admin Menu
0. Exit
1. Add Member
2. Remove Member
3. Update Member Information
4. View All Members
5. Add Trainer
6. Remove Trainer
7. Update Trainer Information
8. View Trainers
9. Add Staff
10. Remove Staff
11. Update Staff Information
12. View Staff
13. Add Subscription Package
14. Remove Subscription Package
15. Update Subscription Package
16. View Subscription Packages
17. Add Schedule
18. Remove Schedule
19. Update Schedule
20. View Schedules
21. Add Private Session
22. Remove Private Session
23. View Private Sessions
24. Mark Attendance
Enter your choice: 12

All Staff
Name: Julkar Junain, Designation: Receptionist, Working Time: 9:00 AM - 5:00 PM

Admin Menu
0. Exit
1. Add Member
2. Remove Member
3. Update Member Information
4. View All Members
5. Add Trainer
6. Remove Trainer
7. Update Trainer Information
8. View Trainers
9. Add Staff
10. Remove Staff
11. Upda